In [1]:
# Cell 1: Configuration
import sys, os, shutil

# === CONFIGURE THESE ===
PARQUET_DIR = "/kaggle/input/datasets/vitorandtxr/cryptobot-binance"
EXPERIMENT  = "multi_pair_v1_kaggle"  # config name under configs/experiments/
RESUME_FROM = None  # set to checkpoint path to resume, e.g. f"{PARQUET_DIR}/best_model.pt"
POS_WEIGHT  = 4  # set to override pos_weight, e.g. 5.0 (None = use config value)
# =======================

DATA_DATASET = PARQUET_DIR
WORK_DIR = "/kaggle/working/SignalCortex"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(f"{DATA_DATASET}/SignalCortex", WORK_DIR)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

# Redirect outputs to persistent Kaggle location
import yaml
config_path = f"configs/experiments/{EXPERIMENT}.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f)
cfg["export"]["output_dir"] = "/kaggle/working/outputs/"
with open(config_path, "w") as f:
    yaml.dump(cfg, f)

print(f"Working dir: {os.getcwd()}")
print(f"Experiment:  {EXPERIMENT}")
print(f"Parquet dir: {PARQUET_DIR}")
print(f"Resume from: {RESUME_FROM}")
print(f"Pos weight:  {POS_WEIGHT or 'from config'}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader').read().strip()}")

Working dir: /kaggle/working/SignalCortex
Experiment:  multi_pair_v1_kaggle
Parquet dir: /kaggle/input/datasets/vitorandtxr/cryptobot-binance
Resume from: None
Pos weight:  4
GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
# Cell 2: Install dependencies
!pip install -q tensorboard tqdm

In [3]:
# Cell 3: Train
resume_flag = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
pw_flag = f"--pw {POS_WEIGHT}" if POS_WEIGHT is not None else ""
!python main.py train \
    --config configs/experiments/{EXPERIMENT}.yaml \
    --parquet {PARQUET_DIR} \
    {resume_flag} {pw_flag}

2026-03-31 16:12:17.740646: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774973537.900025      53 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774973537.943790      53 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774973538.298824      53 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774973538.298867      53 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774973538.298870      53 computation_placer.cc:177] computation placer alr

In [4]:
# Cell 4: Evaluate best model
!python main.py evaluate \
    --config configs/experiments/{EXPERIMENT}.yaml \
    --checkpoint /kaggle/working/outputs/best_model.pt \
    --parquet {PARQUET_DIR}

Traceback (most recent call last):
  File "/kaggle/working/SignalCortex/main.py", line 422, in <module>
    main()
  File "/kaggle/working/SignalCortex/main.py", line 418, in main
    dispatch[args.command](args)
  File "/kaggle/working/SignalCortex/main.py", line 195, in cmd_evaluate
    checkpoint = torch.load(args.checkpoint, map_location=device)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/serialization.py", line 1548, in load
    raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
_pickle.UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code executio

In [5]:
# Cell 5: Backtest per pair
BACKTEST_PAIRS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT"]
for pair in BACKTEST_PAIRS:
    print(f"\n{'='*60}")
    print(f"Backtesting {pair}")
    print(f"{'='*60}")
    !python main.py backtest \
        --config configs/experiments/{EXPERIMENT}.yaml \
        --checkpoint /kaggle/working/outputs/best_model.pt \
        --pair {pair} \
        --parquet {PARQUET_DIR}


Backtesting BTCUSDT
Loading data for BTCUSDT...
Model loaded from /kaggle/working/outputs/best_model.pt (epoch 22)
Running inference on BTCUSDT...

Backtest config: TP=1.44%, SL=0.48%, Trailing=0.5%, Fee=0.075%
Capital: 1000.0 USD | Max drawdown limit: 20.0%
Candles: 443534, BUY prob range: [0.1025, 0.9252]

  Strategy 1: Take Profit / Stop Loss
 Thresh |  Trades | WinRate |  AvgPnL |  TotRet% |  AvgWin |  AvgLoss |     PF |  MaxDD% |  Sharpe |  AvgDur |   Exp% |     Equity |  Status
----------------------------------------------------------------------------------------------------------------------------------
   0.15 |      86 |   19.8% |  -0.250 |   -19.60 |  +1.290 |   -0.630 |   0.50 |  -20.12 | -106.203 |    20.8 |   0.4% |     803.98 |    FAIL
   0.20 |      86 |   19.8% |  -0.250 |   -19.60 |  +1.290 |   -0.630 |   0.50 |  -20.12 | -106.203 |    20.8 |   0.4% |     803.98 |    FAIL
   0.25 |      86 |   19.8% |  -0.250 |   -19.60 |  +1.290 |   -0.630 |   0.50 |  -20.12 | -106

In [6]:
# Cell 6: List saved outputs
for root, dirs, files in os.walk("/kaggle/working/outputs"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"{path} ({size:.1f} MB)")

/kaggle/working/outputs/best_model_normalizer_5m.pkl (0.0 MB)
/kaggle/working/outputs/best_model_normalizer_1h.pkl (0.0 MB)
/kaggle/working/outputs/best_model.pt (11.1 MB)
/kaggle/working/outputs/best_model_normalizer_15m.pkl (0.0 MB)
/kaggle/working/outputs/runs/multiscale_5m/events.out.tfevents.1774973632.fa0f5ab9401f.53.0 (0.0 MB)
/kaggle/working/outputs/runs/multiscale_5m/loss_train/events.out.tfevents.1774974299.fa0f5ab9401f.53.1 (0.0 MB)
/kaggle/working/outputs/runs/multiscale_5m/f1_train/events.out.tfevents.1774974299.fa0f5ab9401f.53.3 (0.0 MB)
/kaggle/working/outputs/runs/multiscale_5m/f1_val/events.out.tfevents.1774974299.fa0f5ab9401f.53.4 (0.0 MB)
/kaggle/working/outputs/runs/multiscale_5m/loss_val/events.out.tfevents.1774974299.fa0f5ab9401f.53.2 (0.0 MB)
